# UFC fight predictor: v3

v2 beat the baseline on the test set, but walk-forward validation showed it winning 2016–2019 and losing every year from 2020 on. This notebook finds out why, fixes it, and builds v3.

The rules from the decision log still apply:
- Only information known **before** the fight is used.
- No betting odds as features. They're only used at the end, to compare against the market.
- Versions are compared with walk-forward validation on the training fights. The test set (June 2023 onward) is scored **once**.

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("archive/ufc-master.csv")
df["date"] = pd.to_datetime(df["date"])

fights = df[df["Winner"].isin(["Red", "Blue"])].copy()
fights["red_win"] = (fights["Winner"] == "Red").astype(int)
fights.shape

(7169, 119)

## 1. The bug that broke v2

The file's ready-made difference columns are not consistent. `age_dif` should be Blue's age minus Red's age, and from mid-2020 on, it is. **Before that, it's Red minus Blue: the sign is flipped.** `loss_dif` and `lose_streak_dif` have the same problem.

The cell below checks, for each year, the share of fights where `age_dif` really equals `B_age - R_age` (skipping fights where both fighters are the same age):

In [2]:
b_minus_r = fights["B_age"] - fights["R_age"]
matches = (fights["age_dif"] == b_minus_r)[b_minus_r != 0]
matches.groupby(fights["date"].dt.year).mean().round(2)

date
2010    0.00
2011    0.00
2012    0.00
2013    0.00
2014    0.00
2015    0.00
2016    0.00
2017    0.00
2018    0.00
2019    0.00
2020    0.75
2021    1.00
2022    1.00
2023    1.00
2024    1.00
2025    1.00
2026    1.00
dtype: float64

That explains the walk-forward pattern. Most training fights come from before 2020, so the model learned "positive `age_dif`" with the old meaning. From 2020 on, the same number means the opposite, so the age weight pushed predictions the wrong way, and the model fell below the baseline.

It also explains the surprising age weight in v1 and v2: the model was averaging two opposite meanings.

**Lesson:** don't trust pre-computed columns. v3 builds every feature from the raw `R_` and `B_` columns.

(`sig_str_dif` has a related problem: when one fighter has no striking stats, it holds made-up values instead of a blank. v3 doesn't use it either.)

## 2. Features, rebuilt from the raw columns

Each stat is calculated per fighter, then turned into a difference. **v3 uses Red minus Blue**, so a positive value means Red has more, and a positive weight means "more of this helps Red." That's easier to read than v2's Blue-minus-Red.

| Feature | Per fighter | Why |
|---|---|---|
| `log_fights` | log of 1 + UFC fights so far | Experience, with diminishing returns: fight 2 vs. 1 matters more than fight 31 vs. 30 |
| `debut` | 1 if this is their first UFC fight | Debut fighters have no UFC stats, so the model needs to know |
| `win_pct` | (wins + 1) / (fights + 2) | Win rate, pulled toward 50% when there are few fights, so 1–0 doesn't count as 100% (the 22–1 vs. 1–0 problem from the baseline) |
| `ko_rate`, `sub_rate` | KO or submission wins / fights | How dangerous they are, as a rate instead of a count |
| `win_streak`, `lose_streak` | Current streaks | Form |
| `age`, `past_prime` | Age, and years past 32 | Age gap, plus extra decline once past their prime |
| `reach` | Reach in cm (a reach of 0 is treated as missing) | Physical edge. Height was dropped because reach already covers it. |
| `sig_str`, `sig_str_pct` | Significant strikes landed per minute, and accuracy | Striking skill |
| `td`, `td_pct` | Takedowns landed, and accuracy | Wrestling skill |
| `sub_att` | Submission attempts | Grappling threat |
| `rank_red`, `rank_blue` | Who's better ranked (blank = neither) | Same as v2 |

Per-fight averages are left **blank for debut fighters**, since there's no UFC history to average. The next section fills those blanks.

In [3]:
def fighter_stats(raw, corner):
    p = corner + "_"
    fights_so_far = raw[p + "wins"] + raw[p + "losses"] + raw[p + "draw"]
    debut = fights_so_far == 0
    has_fights = fights_so_far.where(fights_so_far > 0)   # blank for debuts, so no dividing by 0
    return pd.DataFrame({
        "log_fights": np.log1p(fights_so_far),
        "debut": debut.astype(int),
        "win_pct": (raw[p + "wins"] + 1) / (fights_so_far + 2),
        "ko_rate": raw[p + "win_by_KO/TKO"] / has_fights,
        "sub_rate": raw[p + "win_by_Submission"] / has_fights,
        "win_streak": raw[p + "current_win_streak"],
        "lose_streak": raw[p + "current_lose_streak"],
        "age": raw[p + "age"],
        "past_prime": (raw[p + "age"] - 32).clip(lower=0),
        "reach": raw[p + "Reach_cms"].where(raw[p + "Reach_cms"] > 0),
        "sig_str": raw[p + "avg_SIG_STR_landed"].where(~debut),
        "sig_str_pct": raw[p + "avg_SIG_STR_pct"].where(~debut),
        "td": raw[p + "avg_TD_landed"].where(~debut),
        "td_pct": raw[p + "avg_TD_pct"].where(~debut),
        "sub_att": raw[p + "avg_SUB_ATT"].where(~debut),
    })


def build_features(raw):
    X = fighter_stats(raw, "R") - fighter_stats(raw, "B")
    X.columns = [c + "_diff" for c in X.columns]
    X["rank_red"] = (raw["better_rank"] == "Red").astype(int)
    X["rank_blue"] = (raw["better_rank"] == "Blue").astype(int)
    return X


X_all = build_features(fights)
features = list(X_all.columns)
model_df = pd.concat([fights[["date", "red_win"]], X_all], axis=1)

train = model_df[model_df["date"] <= "2023-06-03"]
test = model_df[model_df["date"] > "2023-06-03"]
print(len(features), "features")
print(train.shape, test.shape)

17 features
(5739, 19) (1430, 19)


## 3. The model is a pipeline

v3 has blanks (debuts, a few missing stats) and features in very different units. A **pipeline** chains three steps so they always run together, in order:

1. `SimpleImputer`: fills each blank with that feature's median
2. `StandardScaler`: puts every feature in the same units (typical-size differences), so the weights can be compared
3. `LogisticRegression`: same model as before

When you call `.fit()` on the pipeline, **all three learn from the training data only**: the medians, the means and spreads, and the weights. When you call `.predict_proba()`, it reuses those training numbers. So the leakage rule from scaling is handled automatically.

In [4]:
def make_model():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=1000))

## 4. Walk-forward validation: v2 vs. v2 fixed vs. v3

Your loop from the last notebook, turned into a function so it can score any version. It only ever sees fights up to the June 2023 cutoff. Three versions:

- **v2 as built:** your v2 features, using the file's broken `age_dif`
- **v2, age fixed:** identical, except `age_dif` is recalculated correctly. This isolates the effect of the bug.
- **v3:** the new features

In [5]:
def walk_forward(feature_cols, data):
    data = data[data["date"] <= "2023-06-03"]          # never touch the test set
    model_scores = []
    base_scores = []
    for i in range(2016, 2024):
        past = data[data["date"].dt.year < i]
        valid = data[data["date"].dt.year == i]
        model = make_model().fit(past[feature_cols], past["red_win"])
        probas = model.predict_proba(valid[feature_cols])[:, 1]
        baseline = [past["red_win"].mean()] * len(valid)
        model_scores.append(log_loss(valid["red_win"], probas))
        base_scores.append(log_loss(valid["red_win"], baseline))
    return pd.Series(model_scores, index=range(2016, 2024)), pd.Series(base_scores, index=range(2016, 2024))


v2_features = ["total_fights_diff", "height_dif", "reach_dif", "win_streak_dif", "rank_red", "rank_blue", "age_dif"]
v2 = fights[["date", "red_win", "height_dif", "reach_dif", "win_streak_dif", "age_dif"]].copy()
v2["total_fights_diff"] = (fights["B_wins"] + fights["B_losses"]) - (fights["R_wins"] + fights["R_losses"])
v2["rank_red"] = X_all["rank_red"]
v2["rank_blue"] = X_all["rank_blue"]

v2_fixed = v2.copy()
v2_fixed["age_dif"] = fights["B_age"] - fights["R_age"]

v2_scores, base_scores = walk_forward(v2_features, v2)
v2_fixed_scores, _ = walk_forward(v2_features, v2_fixed)
v3_scores, _ = walk_forward(features, model_df)

comparison = pd.DataFrame({
    "baseline": base_scores,
    "v2 as built": v2_scores,
    "v2, age fixed": v2_fixed_scores,
    "v3": v3_scores,
})
comparison.loc["average"] = comparison.mean()
comparison.loc["years beating baseline"] = (comparison.drop("average").lt(base_scores, axis=0)).sum()
comparison.round(3)

,baseline,v2 as built,"v2, age fixed",v3
2016,0.676,0.654,0.654,0.656
2017,0.690,0.662,0.662,0.656
2018,0.695,0.671,0.671,0.663
2019,0.683,0.681,0.681,0.677
2020,0.667,0.676,0.651,0.646
2021,0.687,0.696,0.659,0.649
2022,0.675,0.690,0.654,0.633
2023,0.699,0.708,0.664,0.651
average,0.684,0.680,0.662,0.654
years beating baseline,0.000,4.000,8.000,8.000


Fixing the sign alone turns v2 from "wins 4 of 8 years" into "wins all 8." The new features improve on that again, and v3 beats the baseline in every year, including the 2020–2023 years where v2 failed.

Changing the regularization strength `C` (0.01, 0.1 and 1) made almost no difference in validation, so v3 uses the default.

## 5. The final test, opened once

Train v3 on every training fight, then score it on the 1,430 test fights, the same fights v1 and v2 were tested on.

In [6]:
X_train, Y_train = train[features], train["red_win"]
X_test, Y_test = test[features], test["red_win"]

model = make_model().fit(X_train, Y_train)
probs = model.predict_proba(X_test)[:, 1]
train_probs = model.predict_proba(X_train)[:, 1]

base_rate = Y_train.mean()
print("test log loss:    ", round(log_loss(Y_test, probs), 4))
print("train log loss:   ", round(log_loss(Y_train, train_probs), 4))
print("baseline log loss:", round(log_loss(Y_test, [base_rate] * len(Y_test)), 4))
print("test accuracy:    ", round(((probs > 0.5) == Y_test).mean(), 4))
print("always-Red acc:   ", round(Y_test.mean(), 4))

test log loss:     0.6287
train log loss:    0.6515
baseline log loss: 0.6864
test accuracy:     0.6455
always-Red acc:    0.5615


### Calibration: does "70%" mean 70%?

Test fights grouped by v3's predicted chance that Red wins, compared with how often Red actually won:

In [7]:
buckets = pd.cut(probs, [0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1])
pd.DataFrame({"predicted": probs, "red_won": Y_test.to_numpy()}).groupby(buckets, observed=True).agg(
    fights=("red_won", "size"), avg_predicted=("predicted", "mean"), actual=("red_won", "mean")).round(3)

,fights,avg_predicted,actual
"(0.0, 0.3]",40,0.258,0.175
"(0.3, 0.4]",116,0.356,0.328
"(0.4, 0.5]",246,0.455,0.390
"(0.5, 0.6]",378,0.553,0.526
"(0.6, 0.7]",377,0.647,0.663
"(0.7, 0.8]",216,0.745,0.759
"(0.8, 1.0]",57,0.837,0.860


### Against the betting market

The odds were kept out of the model so this comparison would mean something. American odds convert to probabilities: −150 means 150/250 = 60%, and +130 means 100/230 = 43.5%. The two sides add up to a bit over 100% (the bookmaker's cut), so they're rescaled to sum to 100%. Some recent fights have no odds in the file, so this compares only the fights that do.

In [8]:
def implied(odds):
    with np.errstate(divide="ignore"):    # np.where calculates both formulas for every row
        return np.where(odds < 0, -odds / (-odds + 100), 100 / (odds + 100))

red_imp = implied(fights.loc[test.index, "R_odds"].to_numpy())
blue_imp = implied(fights.loc[test.index, "B_odds"].to_numpy())
market = red_imp / (red_imp + blue_imp)
has_odds = ~np.isnan(market)

print("test fights with odds:", has_odds.sum())
print("log loss  v3:", round(log_loss(Y_test[has_odds], probs[has_odds]), 4),
      " market:", round(log_loss(Y_test[has_odds], market[has_odds]), 4),
      " baseline:", round(log_loss(Y_test[has_odds], [base_rate] * has_odds.sum()), 4))
print("accuracy  v3:", round(((probs[has_odds] > 0.5) == Y_test[has_odds]).mean(), 4),
      " market:", round(((market[has_odds] > 0.5) == Y_test[has_odds]).mean(), 4))

test fights with odds: 1207
log loss  v3: 0.6214  market: 0.5814  baseline: 0.6867
accuracy  v3: 0.6479  market: 0.6993


### What v3 learned

Features are scaled, so these weights can be compared directly. Positive means "more of this for Red helps Red."

In [9]:
weights = pd.Series(model[-1].coef_[0], index=features)
weights.sort_values(key=abs, ascending=False).round(3)

age_diff           -0.255
td_diff             0.191
win_pct_diff        0.108
win_streak_diff     0.099
reach_diff          0.092
rank_red            0.087
past_prime_diff    -0.076
sig_str_pct_diff    0.072
debut_diff         -0.062
rank_blue          -0.059
sig_str_diff        0.059
sub_rate_diff       0.027
td_pct_diff        -0.025
ko_rate_diff        0.025
lose_streak_diff   -0.018
log_fights_diff     0.016
sub_att_diff       -0.010
dtype: float64

## 6. Predicting the next event

`archive/upcoming.csv` holds the next card after the data ends (April 4, 2026). For real predictions, the model is refit on **all** fights, test set included. The test set's job (an honest score) is done, and the newest fights are the most useful ones to learn from.

In [10]:
final_model = make_model().fit(model_df[features], model_df["red_win"])

upcoming = pd.read_csv("archive/upcoming.csv")
p_red = final_model.predict_proba(build_features(upcoming)[features])[:, 1]
up_red = implied(upcoming["R_odds"].to_numpy())
up_blue = implied(upcoming["B_odds"].to_numpy())

pd.DataFrame({
    "red": upcoming["R_fighter"],
    "blue": upcoming["B_fighter"],
    "v3 red %": (p_red * 100).round(0),
    "v3 blue %": ((1 - p_red) * 100).round(0),
    "market red %": (up_red / (up_red + up_blue) * 100).round(0),
})

,red,blue,v3 red %,v3 blue %,market red %
0,Renato Moicano,Chris Duncan,36.0,64.0,38.0
1,Virna Jandiroba,Tabatha Ricci,47.0,53.0,52.0
2,Abdul Rakhman Yakhyaev,Brendson Ribeiro,96.0,4.0,90.0
3,Ethyn Ewing,Rafael Estevam,39.0,61.0,57.0
4,Tommy McMillen,Manolo Zecchini,77.0,23.0,89.0
5,Jose Delano,Robert Ruchala,65.0,35.0,74.0
6,Guilherme Pat,Thomas Petersen,57.0,43.0,52.0
7,Alessandro Costa,Stewart Nicoll,61.0,39.0,76.0
8,Lando Vannata,Darrius Flowers,52.0,48.0,66.0
9,Alice Pereira,Hailey Cowan,69.0,31.0,53.0


## 7. Does the model add anything the market doesn't know?

The odds stayed out of v3 so its score would show what fighter stats can predict on their own. This section asks a different question: **if the model is given the odds, do the stats still help?**

Setup:
- The odds become the market's probability (as in section 5), then **log-odds**: `log(p / (1 - p))`. That's the same scale as logistic regression's score, so the model can use the market's opinion directly.
- **Odds only:** logistic regression with just the market's log-odds. It can slightly recalibrate the market (for example, adjust for the red-corner effect), so it's the fair opponent.
- **Odds + stats:** the market's log-odds plus v3's 17 features.

Both models train on training fights that have odds (5,709 of 5,739) and are tested on the 1,207 test fights that have odds. **Any gain from odds-only to odds + stats comes from your features.**

This reuses the test set for a new question, decided before looking at the results. v3 itself doesn't change.

In [11]:
red_imp_all = implied(fights["R_odds"].to_numpy())
blue_imp_all = implied(fights["B_odds"].to_numpy())
market = pd.Series(red_imp_all / (red_imp_all + blue_imp_all), index=fights.index)
odds_df = model_df.assign(market_logit=np.log(market / (1 - market)))
odds_df = odds_df[np.isfinite(odds_df["market_logit"])]

o_train = odds_df[odds_df["date"] <= "2023-06-03"]
o_test = odds_df[odds_df["date"] > "2023-06-03"]
odds_plus = ["market_logit"] + features


def make_odds_model():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))


odds_model = make_odds_model().fit(o_train[["market_logit"]], o_train["red_win"])
both_model = make_model().fit(o_train[odds_plus], o_train["red_win"])

y_odds = o_test["red_win"]
candidates = {
    "market as-is": 1 / (1 + np.exp(-o_test["market_logit"])),
    "odds only": odds_model.predict_proba(o_test[["market_logit"]])[:, 1],
    "odds + stats": both_model.predict_proba(o_test[odds_plus])[:, 1],
    "v3 (stats only)": model.predict_proba(o_test[features])[:, 1],
}
pd.DataFrame({
    "log loss": {k: log_loss(y_odds, p) for k, p in candidates.items()},
    "accuracy": {k: ((p > 0.5) == y_odds).mean() for k, p in candidates.items()},
}).round(4)

,log loss,accuracy
market as-is,0.5814,0.6993
odds only,0.5793,0.7051
odds + stats,0.5747,0.7084
v3 (stats only),0.6214,0.6479


The weights, scaled as before. The market's opinion carries most of the weight, as expected. What's left over shows where your stats disagree with it.

In [12]:
pd.Series(both_model[-1].coef_[0], index=odds_plus).sort_values(key=abs, ascending=False).round(3).head(10)

market_logit       0.769
td_diff            0.135
age_diff          -0.110
win_pct_diff      -0.102
win_streak_diff    0.066
reach_diff         0.058
sub_att_diff       0.052
rank_blue         -0.042
td_pct_diff       -0.036
sub_rate_diff     -0.032
dtype: float64

### Is the improvement real?

The gain is small, so it needs two checks.

**1. Walk-forward:** does adding the stats help in most years, or just on the test split?

In [13]:
rows = []
for i in range(2016, 2024):
    past = o_train[o_train["date"].dt.year < i]
    valid = o_train[o_train["date"].dt.year == i]
    p_odds = make_odds_model().fit(past[["market_logit"]], past["red_win"]).predict_proba(valid[["market_logit"]])[:, 1]
    p_both = make_model().fit(past[odds_plus], past["red_win"]).predict_proba(valid[odds_plus])[:, 1]
    rows.append({"year": i, "odds only": log_loss(valid["red_win"], p_odds),
                 "odds + stats": log_loss(valid["red_win"], p_both)})

odds_wf = pd.DataFrame(rows).set_index("year")
odds_wf.loc["average"] = odds_wf.mean()
print("years where stats helped:", (odds_wf.drop("average")["odds + stats"] < odds_wf.drop("average")["odds only"]).sum(), "of 8")
odds_wf.round(4)

years where stats helped: 6 of 8


,odds only,odds + stats
year,,
2016,0.6225,0.6239
2017,0.6125,0.6025
2018,0.6003,0.5999
2019,0.6319,0.6350
2020,0.6234,0.6176
2021,0.6283,0.6175
2022,0.5977,0.5938
2023,0.5939,0.5847
average,0.6138,0.6094


**2. Bootstrap on the test set:** with 1,207 fights, how much could the gain move just from which fights happened to be in the test set? The bootstrap resamples the test fights 2,000 times (drawing fights at random, with repeats allowed), recomputes the improvement each time, and reports the range covering the middle 95% of results. If that range includes 0, the test set can't rule out "no real improvement."

In [14]:
def per_fight_loss(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).to_numpy()


gain = per_fight_loss(y_odds, candidates["odds only"]) - per_fight_loss(y_odds, candidates["odds + stats"])
rng = np.random.default_rng(0)
resampled = [gain[rng.integers(0, len(gain), len(gain))].mean() for _ in range(2000)]
print("improvement in log loss:", round(gain.mean(), 4))
print("95% range:", np.percentile(resampled, [2.5, 97.5]).round(4))

improvement in log loss: 0.0045
95% range: [-0.0016  0.0111]


### Verdict

Adding your stats to the odds lowers log loss by about 0.004–0.005: on the test set, and on average across walk-forward years, where it helped in 6 of 8 years. That's a consistent direction, but a small effect. The test-set range includes 0, so it **can't be called proven** with this much data.

So, a fair summary: **the market already prices in almost everything this file knows.** There may be a small amount of extra signal, mostly in wrestling (`td_diff`) and age. After accounting for the odds, a better record (`win_pct_diff`) slightly *hurts*, which suggests the market may overrate records a little. Those are hypotheses worth testing with more data, not findings.

This is not evidence of a betting edge. A gain this small, which isn't even certain, wouldn't cover the bookmaker's cut.

## Results

| | v1 | v2 | **v3** | Baseline | Betting market* |
|---|---|---|---|---|---|
| Walk-forward log loss (2016–2023) | — | 0.680 (beats baseline 4/8 years) | **0.654 (8/8 years)** | 0.684 | — |
| Test log loss | 0.704 | 0.678 | **0.629** | 0.686 | 0.581 |
| Test accuracy | 55.9% | 57.0% | **64.6%** | 56.2% (always Red) | 69.9% |

\*The market is compared on the 1,207 test fights that have odds. On those same fights, v3 scores 0.621 log loss and 64.8% accuracy.

**Market + model (section 7), same 1,207 fights:**

| | Log loss | Accuracy |
|---|---|---|
| Odds only | 0.579 | 70.5% |
| Odds + v3's features | 0.575 | 70.8% |

**What this means:**
- v3 clearly beats the baseline, consistently across years. It's not a lucky split.
- Calibration is good: when v3 says 60–70%, Red wins about 66% of the time. Its weakest spot is fights it calls 40–50% for Red, where Red wins less often than predicted.
- The betting market is still well ahead, which is expected. It knows things this file doesn't: injuries, camp changes, pre-UFC records, styles.
- Given the odds, v3's features add a small improvement that holds in 6 of 8 validation years, but it's too small to be certain with this data. The market already prices in nearly everything here.
- The biggest jump came from **fixing the data, not from a better model**: the same v2 features with the age sign corrected went from losing to the baseline to beating it every year.

**Ideas for v4:**
- Tree models (random forest, gradient boosting) can pick up effects a weighted sum can't, like age mattering more past 35, or reach mattering more for strikers.
- Pre-UFC experience: debut fighters currently look like total beginners.
- The corner question: the model leans on Red's built-in advantage, so for a matchup with no assigned corners, you could predict both ways and average.